In [ ]:
# dropout 
# during each forward pass p ,raandomly kill some neurons and during 
# inference then use all the neurons but scale up the outputs by (1-p) 

# either you can scale down  the inference outputs by (1-P) 
# inverted dropout : or scale up the training outputs by 1/(1-p), so inference needs no cahnge at all 


# Why inverted dropout wins: You don't want inference code to 
# know about dropout rates. Inference should just be a clean forward pass 
# — no special logic. Simpler, faster, less error-prone.
import torch 

def dropout(x, p, training): 
    if training == True: 
        mask = (torch.rand_like(x) > p ).float()
        # randomly zero p of these 
        out = mask *x/ (1-p)
    else: 
        out = x 
    return out 
# torch.rand_like(x) — uniform distribution between [0, 1). Every value equally likely. Good for "coin flip" style masks.

# torch.randn_like(x) — standard normal distribution (mean=0, std=1)

In [ ]:
def attention_with_dropout(Q, K, V, p, training): 
    dk = Q.shape[-1]
    attention = Q@K.transpose(-1,-2) / torch.sqrt(dk )
    scores = torch.softmax(attention, dim = -1)
    

    if training == True :  
        dropout = (torch.rand_like(scores) > p).float()
        scores   = dropout*scores 
        scores = scores / (1-p)     
    out = scores @ V 
    return out 


# Dropout — Revision Notes

## Why dropout?
- Big model + limited data = overfitting (memorizes training examples)
- Dropout forces neurons to be useful independently — prevents co-adaptation
- Each forward pass trains a different sub-network (ensemble effect)

## Inverted Dropout
- Training: `out = x * mask / (1 - p)` — scale UP survivors
- Inference: `out = x` — do nothing
- Why inverted? Inference stays a clean forward pass, no need to know dropout rate

## torch.rand_like vs torch.randn_like
- `rand_like` = uniform [0, 1) — use for dropout masks
- `randn_like` = normal N(0,1) — use for noise sampling (VAEs, diffusion)
- The **n** stands for **normal**

## Where dropout is applied in a Transformer (3 places per block)
```
# Attention sub-layer
scores = softmax(Q @ K^T / sqrt(d_k))
scores = dropout(scores)              # (1) attention dropout — drops token-to-token connections
attn_out = scores @ V
x = x + dropout(attn_out)            # (2) residual dropout — drops dimensions of attention output

# FFN sub-layer
x = x + dropout(ffn(x))              # (3) residual dropout — drops dimensions of FFN output
```

## Why dropout on scores AFTER softmax (not before)?
- After softmax: zeroing an entry = truly removing that attention connection. Clean removal.
- Before softmax: zeroing a logit sets it to 0, but softmax gives exp(0)=1 relative weight — token still participates with wrong score. Noisy corruption, not removal.
- To truly mask before softmax, you'd use -inf (like padding masks), not 0.

## Residual stream is NEVER dropped
- The residual path (x = x + ...) is the gradient highway
- Dropout only applies to the "delta" being added, not to x itself
- Gradients always flow through the addition unmodified

## Why large models (GPT-3, LLaMA) use NO dropout
- So much training data that the model never memorizes
- Overfitting isn't the failure mode — underfitting is
- Dropout would just slow learning and waste capacity
- Weight decay alone suffices for regularization

## Dropout hyperparameter
- Transformers: p = 0.1 to 0.2 (small — 36 dropout points across 12 blocks would kill signal otherwise)
- Old CNNs: p = 0.5 (fewer layers, more aggressive)
- With heavy data augmentation: dropout can be redundant

## Overfitting checklist (in order)
1. More data / data augmentation
2. Weight decay (always on, tune λ)
3. Dropout (add if still overfitting)
4. Reduce model size (last resort)

# Dropout — Q&A for Revision

## Q1: If p=0.3 and input has 10 neurons, how many survive? What's the scaling factor?
- **Survivors:** 7 on average (10 * (1-0.3))
- **Scaling factor:** 1/(1-p) = 1/0.7 ≈ 1.43
- **Why?** So expected output sum stays the same: 7 neurons * 1.43 ≈ 10 (as if no dropout)
- Scaling is based on drop probability, NOT on count of survivors

## Q2: What is E[dropout(x)] during training?
- E[dropout(x)] = x, regardless of p
- The scaling by 1/(1-p) ensures this invariant
- Each neuron: prob (1-p) of being kept and scaled by 1/(1-p), prob p of being 0
- E = (1-p) * x/(1-p) + p * 0 = x

## Q3: Your colleague puts dropout AFTER softmax on attention weights. What's being dropped?
- You're dropping attention connections — randomly zeroing token-to-token links
- Forces each token to not always rely on the same set of other tokens
- Makes attention patterns more distributed and robust

## Q4: What goes wrong if dropout is applied BEFORE softmax (on raw logits)?
- Zeroing a logit before softmax does NOT remove that connection
- softmax(0) = exp(0) = 1 relative weight — token STILL participates with neutral score
- It corrupts the distribution unpredictably rather than cleanly removing connections
- To truly mask before softmax, use -inf (like padding masks), not 0
- That's masking, not dropout — different mechanism

## Q5: Attention computation order
```
logits = Q @ K^T / sqrt(d_k)     ← scale BEFORE softmax (prevents saturation)
scores = softmax(logits)          ← normalize to probabilities
scores = scores * mask / (1-p)    ← dropout on SCORES (training only)
output = scores @ V               ← weighted sum of values
```
- sqrt(d_k) prevents dot products from being too large (variance grows with d_k)
- If you scale after softmax, the damage is already done (softmax already saturated)

## Q6: Where is dropout applied in a full Transformer block? (3 places)
1. **Attention dropout** — on softmax scores (drops which tokens attend to which)
2. **Residual dropout after attention** — on attention sub-layer output, before adding to residual
3. **Residual dropout after FFN** — on FFN output, before adding to residual

```python
# Full transformer block with dropout
scores = softmax(Q @ K.T / sqrt(d_k))
scores = dropout(scores)                # (1) attention dropout
attn_out = scores @ V

x = x + dropout(attn_out)              # (2) residual dropout after attention
x = layernorm(x)

ffn_out = W2 @ relu(W1 @ x)
x = x + dropout(ffn_out)               # (3) residual dropout after FFN
x = layernorm(x)
```

Key: residual stream (x itself) is NEVER dropped — only the deltas being added to it.

## Q7: With p=0.1 across 12 blocks (36 dropout points), what fraction of signal survives?
- Naively: 0.9^36 ≈ 0.02 (only 2%)
- BUT this is misleading — the residual connection bypasses all dropout
- Information flows through x = x + dropout(delta), so even if delta is partially dropped, x carries the history
- This is why p must stay small (0.1-0.2) in deep Transformers — even small p compounds

## Q8: Why do GPT-3 / LLaMA use NO dropout?
- Trained on hundreds of billions of tokens — model rarely sees same data twice
- Can't memorize when data is effectively infinite
- Overfitting isn't the problem — underfitting is
- Dropout would slow convergence and waste capacity for no regularization benefit
- Weight decay alone keeps weights well-behaved

## Q9: Weight Decay vs Dropout — when to use which?

| | Dropout | Weight Decay |
|---|---------|-------------|
| What | Zeros random neurons during training | Shrinks all weights toward zero each step |
| Applied to | Activations (hidden layers) | Parameters (weights) |
| Effect | Prevents co-adaptation between neurons | Prevents large weights, simpler solutions |
| Typical value | p=0.1-0.3 (Transformers), 0.5 (old CNNs) | λ=0.01-0.1 |
| Redundant when | Heavy data augmentation / massive data | Model already small |
| Used together? | Yes — complementary mechanisms |

## Q10: Overfitting — what to try (in order)
1. More data / better data augmentation (most effective)
2. Weight decay (always on, just tune λ — no training cost)
3. Dropout (add if still overfitting — slight training cost)
4. Early stopping (monitor val loss)
5. Reduce model size (last resort — loses capacity)